In [ ]:
import pandas as pd
from plotnine import *

from vpop_calibration.api import *

%load_ext autoreload
%autoreload 2

In [ ]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    inputs=[
        "ka",
        "V",
        "Vm",
        "Km",
        "alpha",
        "Beta",
        "Dose",
        "gamma",
    ],
    outputs=["C", "cumulative_hazard", "log_hazard"],
    path_to_solving_options="sv.json",
)

protocol_design = pd.DataFrame({"protocol_arm": ["arm-A", "arm-B"], "Dose": [50, 100]})
struct_model = StructuralSimwork(protocol_design = protocol_design, model= model)

In [ ]:
input_params = {
    "model_intrinsic": {
        "gamma": {"prior": 15.0, "constraint": {"low": 0.0}},
    },
    "pdu": {
        "ka": {"prior": 0.5,  "prior_omega": 0.3},
        "V":  {"prior": 70.0, "prior_omega": 0.2},
        "Vm": {"prior": 6.0,  "prior_omega": 0.1},
        "Km": {"prior": 0.2,  "prior_omega": 0.2},
        "Beta":  {"prior": 0.5, "prior_omega": 0.1},
    },
    "error_model": {"C": {"error_type": "combined", "sigma_add": 0.05, "sigma_prop": 0.05}},
    "pdk": [],
    "time_to_event": {
        "hazard_name": "hazard",
        "coefficients": {
            "alpha": {"prior": 1.0},
        },
    },
}
training_df = pd.read_csv("data.csv")

In [ ]:
config = Config(
    saem=SaemConfigDict(
        fixed_effects_nb_iter=10,
        nb_iter_burnin=0,
        nb_iter_smoothing=50,
        nb_iter_learning=50,
        plot_frames=1,
        plot_columns=5,
    )
)

In [ ]:
nlme_model = NlmeModel(
    df=training_df, structural_model=struct_model, input_params=input_params, config=config
)

In [ ]:
nlme_model.optimizer.run()